> **Public release note.** Notebook outputs have been removed because the underlying Malaysian Motor claims data are confidential. Execution counts have also been cleared to provide clean public versions of the notebooks. Local user-specific paths and individual claim identifiers have been removed. The repository documents the data-processing, modelling and validation workflow used in the dissertation, but the numerical results cannot be reproduced end-to-end without the confidential input data.


# 05.8A — XGBoost Future-Development Model

**Purpose:** fit XGBoost to exactly the same leakage-controlled inputs,
future-development target and held-out test observations as the
Random Forest.

The notebook answers five empirical questions:

1. Does XGBoost reduce the DEV_QTR_4 overprediction?
2. Does it reduce the severe DEV_QTR_8 overprediction?
3. Does it improve claim-level error for severe outcomes?
4. Does it reduce accident-year WAPE?
5. Does it improve total calibration against the later observed incurred
   BI Excess position?

The held-out 2020 Q1–2022 Q4 test observations are not used for model
selection. The final four valuation quarters within the training period are
used as a temporal validation window for early stopping and focused
hyperparameter selection.

This cell sets up the XGBoost notebook. It loads the needed Python packages, makes sure XGBoost is installed, sets the project, input, and output folders, and reads the feature_manifest.json from the first notebook. This way, XGBoost uses the same snapshots, features, and targets as the Random Forest.

It also prints the Python and package versions and checks that the needed folders are there. This helps make sure the setup is correct and reproducible before fitting the XGBoost models.

In [ ]:
from pathlib import Path
import json
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

try:
    import xgboost as xgb
    from xgboost import XGBRegressor
except ImportError as exc:
    raise ImportError(
        "XGBoost is not installed in this environment. Install it in the "
        "active environment before rerunning this notebook."
    ) from exc

PROJECT_FOLDER = Path(
    "/path/to/BI_large_claims_project"
)

INPUT_FOLDER = (
    PROJECT_FOLDER
    / "processed"
    / "chapter5_outputs"
    / "section_5_4A_clean_snapshot_model_inputs"
)

RF_OUTPUT_FOLDER = (
    PROJECT_FOLDER
    / "processed"
    / "chapter5_outputs"
    / "section_5_7A_clean_rf_future_development"
)

OUTPUT_FOLDER = (
    PROJECT_FOLDER
    / "processed"
    / "chapter5_outputs"
    / "section_5_8A_clean_xgboost_future_development"
)
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

with open(INPUT_FOLDER / "feature_manifest.json") as f:
    manifest = json.load(f)

SNAPSHOTS = manifest["snapshots"]
FEATURE_COLUMNS = manifest["feature_columns"]
CATEGORICAL_FEATURES = manifest["categorical_features"]
NUMERIC_FEATURES = manifest["numeric_features"]
TARGET_LATEST = manifest["target_latest"]
TARGET_FUTURE = manifest["target_future"]
SNAPSHOT_EXCESS = manifest["snapshot_excess"]

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgb.__version__)
print("Input folder exists:", INPUT_FOLDER.exists())
print("RF output folder exists:", RF_OUTPUT_FOLDER.exists())
print("Output folder:", OUTPUT_FOLDER)

## Focused candidate set

The candidate set is deliberately small. The purpose is to test whether
sequential boosting improves the reserving result, not to conduct an
unrestricted hyperparameter search.

Candidate selection is based on the temporal validation window:

1. lowest accident-year WAPE;
2. if effectively tied, lowest absolute portfolio bias.

Early stopping determines the number of boosting rounds. The selected
number of rounds is then used to refit the model on the full training
period before the untouched test set is evaluated.

In [ ]:
BASE_PARAMS = {
    "objective": "reg:squarederror",
    "booster": "gbtree",
    "tree_method": "hist",
    "n_estimators": 2000,
    "early_stopping_rounds": 75,
    "eval_metric": "rmse",
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": 0,
}

CANDIDATES = [
    {
        "candidate": "XGB_A_depth3",
        "learning_rate": 0.03,
        "max_depth": 3,
        "min_child_weight": 50,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "reg_alpha": 0.0,
        "reg_lambda": 10.0,
        "gamma": 0.0,
    },
    {
        "candidate": "XGB_B_depth4",
        "learning_rate": 0.03,
        "max_depth": 4,
        "min_child_weight": 50,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "reg_alpha": 1.0,
        "reg_lambda": 10.0,
        "gamma": 0.0,
    },
    {
        "candidate": "XGB_C_depth5",
        "learning_rate": 0.025,
        "max_depth": 5,
        "min_child_weight": 100,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "reg_alpha": 1.0,
        "reg_lambda": 20.0,
        "gamma": 0.0,
    },
    {
        "candidate": "XGB_D_conservative",
        "learning_rate": 0.03,
        "max_depth": 4,
        "min_child_weight": 200,
        "subsample": 0.90,
        "colsample_bytree": 0.90,
        "reg_alpha": 5.0,
        "reg_lambda": 20.0,
        "gamma": 0.0,
    },
]

with open(OUTPUT_FOLDER / "xgb_candidate_parameters.json", "w") as f:
    json.dump(
        {"base_parameters": BASE_PARAMS, "candidates": CANDIDATES},
        f,
        indent=2,
    )

display(pd.DataFrame(CANDIDATES))

This section explains the reusable helper functions that XGBoost needs before fitting the models. It covers how to handle missing numeric and categorical values, how to one-hot encode categorical variables, how to load snapshot files and check them for leakage or misalignment, and how to create a time-based validation split using the most recent four valuation quarters.

It also introduces the main evaluation tools. These include reconstructing the projected most recent BI Excess figure from the predicted future development, summarizing results by accident year, safely calculating WAPE and bias, and separately assessing performance for the more severe claims.

In [ ]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def build_preprocessor():
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_one_hot_encoder()),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_transformer, NUMERIC_FEATURES),
            ("categorical", categorical_transformer, CATEGORICAL_FEATURES),
        ],
        remainder="drop",
    )


def safe_ratio(numerator, denominator):
    if denominator == 0:
        return np.nan
    return numerator / denominator


def wape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denominator = np.abs(y_true).sum()
    if denominator == 0:
        return np.nan
    return np.abs(y_true - y_pred).sum() / denominator


def load_snapshot(snapshot):
    folder = INPUT_FOLDER / snapshot

    X_train = pd.read_parquet(folder / "X_train.parquet")
    X_test = pd.read_parquet(folder / "X_test.parquet")
    targets_train = pd.read_parquet(folder / "targets_train.parquet")
    targets_test = pd.read_parquet(folder / "targets_test.parquet")
    meta_train = pd.read_parquet(folder / "meta_train.parquet")
    meta_test = pd.read_parquet(folder / "meta_test.parquet")

    if list(X_train.columns) != FEATURE_COLUMNS:
        raise ValueError(f"{snapshot}: X_train differs from feature manifest.")
    if list(X_test.columns) != FEATURE_COLUMNS:
        raise ValueError(f"{snapshot}: X_test differs from feature manifest.")

    if len(X_train) != len(targets_train) or len(X_train) != len(meta_train):
        raise ValueError(f"{snapshot}: training files are not row-aligned.")
    if len(X_test) != len(targets_test) or len(X_test) != len(meta_test):
        raise ValueError(f"{snapshot}: test files are not row-aligned.")

    forbidden_present = set(manifest["forbidden_features"]).intersection(
        X_train.columns
    )
    if forbidden_present:
        raise ValueError(
            f"{snapshot}: forbidden predictors found: {sorted(forbidden_present)}"
        )

    return X_train, X_test, targets_train, targets_test, meta_train, meta_test


def temporal_validation_masks(meta_train, quarter_count=4):
    if "VALUATION_QTR_INDEX" not in meta_train.columns:
        raise ValueError("VALUATION_QTR_INDEX is required for temporal validation.")

    quarter_values = np.sort(
        pd.to_numeric(
            meta_train["VALUATION_QTR_INDEX"], errors="coerce"
        ).dropna().unique()
    )

    if len(quarter_values) <= quarter_count:
        raise ValueError(
            f"Only {len(quarter_values)} valuation quarters are available; "
            f"cannot reserve {quarter_count} quarters for validation."
        )

    validation_quarters = quarter_values[-quarter_count:]
    validation_mask = meta_train["VALUATION_QTR_INDEX"].isin(validation_quarters)
    model_train_mask = ~validation_mask

    if model_train_mask.sum() == 0 or validation_mask.sum() == 0:
        raise ValueError("Temporal training or validation partition is empty.")

    return model_train_mask.to_numpy(), validation_mask.to_numpy(), validation_quarters


def reconstruct_latest(snapshot_amount, predicted_future):
    raw_latest = np.asarray(snapshot_amount, dtype=float) + np.asarray(
        predicted_future, dtype=float
    )
    deployed_latest = np.maximum(raw_latest, 0.0)
    deployed_future = deployed_latest - np.asarray(snapshot_amount, dtype=float)
    return raw_latest, deployed_latest, deployed_future


def accident_year_summary(
    meta,
    snapshot_amount,
    actual_future,
    actual_latest,
    predicted_future_raw,
    predicted_latest,
):
    df = meta.reset_index(drop=True).copy()
    df["actual_snapshot_bixs"] = np.asarray(snapshot_amount, dtype=float)
    df["actual_future_development"] = np.asarray(actual_future, dtype=float)
    df["actual_latest_bixs"] = np.asarray(actual_latest, dtype=float)
    df["predicted_future_development_raw"] = np.asarray(
        predicted_future_raw, dtype=float
    )
    df["predicted_latest_bixs"] = np.asarray(predicted_latest, dtype=float)

    summary = (
        df.groupby("ACC_YEAR", as_index=False)
        .agg(
            claim_count=("CLAIMS_KEY", "count"),
            actual_snapshot_bixs=("actual_snapshot_bixs", "sum"),
            actual_future_development=("actual_future_development", "sum"),
            predicted_future_development_raw=(
                "predicted_future_development_raw", "sum"
            ),
            actual_latest_bixs=("actual_latest_bixs", "sum"),
            predicted_latest_bixs=("predicted_latest_bixs", "sum"),
        )
    )

    summary["latest_error"] = (
        summary["predicted_latest_bixs"] - summary["actual_latest_bixs"]
    )
    summary["latest_absolute_error"] = summary["latest_error"].abs()
    summary["latest_percentage_error"] = np.where(
        summary["actual_latest_bixs"] != 0,
        summary["latest_error"] / summary["actual_latest_bixs"],
        np.nan,
    )
    summary["latest_bias_ratio"] = np.where(
        summary["actual_latest_bixs"] != 0,
        summary["predicted_latest_bixs"] / summary["actual_latest_bixs"],
        np.nan,
    )

    ay_wape = safe_ratio(
        summary["latest_absolute_error"].sum(),
        summary["actual_latest_bixs"].abs().sum(),
    )
    return summary, ay_wape


def severe_metrics(y_true, y_pred, severe_threshold):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mask = y_true >= severe_threshold
    count = int(mask.sum())

    if count == 0:
        return {
            "severe_claim_count": 0,
            "severe_future_mae": np.nan,
            "severe_future_rmse": np.nan,
            "severe_future_wape": np.nan,
        }

    return {
        "severe_claim_count": count,
        "severe_future_mae": mean_absolute_error(y_true[mask], y_pred[mask]),
        "severe_future_rmse": np.sqrt(
            mean_squared_error(y_true[mask], y_pred[mask])
        ),
        "severe_future_wape": wape(y_true[mask], y_pred[mask]),
    }

## Temporal validation and candidate selection

The validation set consists of the latest four valuation quarters within the
original training period. This retains the out-of-time structure:

- model-selection training: earlier valuation quarters;
- validation: latest four pre-2020 quarters;
- final test: 2020 Q1–2022 Q4, untouched until final evaluation.

This is the final reporting and export step for the XGBoost notebook. It turns the validation results, final test results and accident-year results into tables, saves them as CSV files, and stores the selected XGBoost model settings in a JSON file.

In [ ]:
validation_results = []
selected_models = {}
final_test_results = []
final_ay_results = []
severe_thresholds = {}

for snapshot in SNAPSHOTS:
    print("\n" + "=" * 80)
    print("Snapshot:", snapshot)

    (
        X_train_full,
        X_test,
        targets_train_full,
        targets_test,
        meta_train_full,
        meta_test,
    ) = load_snapshot(snapshot)

    model_train_mask, validation_mask, validation_quarters = (
        temporal_validation_masks(meta_train_full, quarter_count=4)
    )

    X_model_train = X_train_full.loc[model_train_mask].reset_index(drop=True)
    X_validation = X_train_full.loc[validation_mask].reset_index(drop=True)

    targets_model_train = (
        targets_train_full.loc[model_train_mask].reset_index(drop=True)
    )
    targets_validation = (
        targets_train_full.loc[validation_mask].reset_index(drop=True)
    )

    meta_model_train = meta_train_full.loc[model_train_mask].reset_index(drop=True)
    meta_validation = meta_train_full.loc[validation_mask].reset_index(drop=True)

    y_model_train = targets_model_train[TARGET_FUTURE].astype(float).to_numpy()
    y_validation = targets_validation[TARGET_FUTURE].astype(float).to_numpy()

    validation_snapshot_amount = (
        targets_validation[SNAPSHOT_EXCESS].astype(float).to_numpy()
    )
    validation_actual_latest = (
        targets_validation[TARGET_LATEST].astype(float).to_numpy()
    )

    positive_training_targets = y_model_train[y_model_train > 0]
    if len(positive_training_targets) == 0:
        raise ValueError(f"{snapshot}: no positive training targets.")

    severe_threshold = float(np.quantile(positive_training_targets, 0.90))
    severe_thresholds[snapshot] = severe_threshold

    print(
        "Model-selection train rows:", len(X_model_train),
        "| validation rows:", len(X_validation),
        "| validation quarters:", validation_quarters.tolist(),
        "| severe threshold:", f"{severe_threshold:,.2f}",
    )

    preprocessor_selection = build_preprocessor()
    X_model_train_processed = preprocessor_selection.fit_transform(X_model_train)
    X_validation_processed = preprocessor_selection.transform(X_validation)

    candidate_rows = []
    candidate_objects = {}

    for candidate in CANDIDATES:
        candidate_name = candidate["candidate"]
        candidate_params = {
            key: value for key, value in candidate.items()
            if key != "candidate"
        }

        model = XGBRegressor(
            **BASE_PARAMS,
            **candidate_params,
        )

        model.fit(
            X_model_train_processed,
            y_model_train,
            eval_set=[(X_validation_processed, y_validation)],
            verbose=False,
        )

        predicted_validation_future = model.predict(X_validation_processed)
        (
            validation_latest_raw,
            validation_latest_deployed,
            validation_future_deployed,
        ) = reconstruct_latest(
            validation_snapshot_amount,
            predicted_validation_future,
        )

        validation_ay, validation_ay_wape = accident_year_summary(
            meta_validation,
            validation_snapshot_amount,
            y_validation,
            validation_actual_latest,
            predicted_validation_future,
            validation_latest_deployed,
        )

        validation_latest_total = float(validation_actual_latest.sum())
        predicted_validation_latest_total = float(
            validation_latest_deployed.sum()
        )
        validation_bias_ratio = safe_ratio(
            predicted_validation_latest_total,
            validation_latest_total,
        )

        severe_result = severe_metrics(
            y_validation,
            predicted_validation_future,
            severe_threshold,
        )

        best_iteration = getattr(model, "best_iteration", None)
        if best_iteration is None:
            selected_rounds = int(BASE_PARAMS["n_estimators"])
        else:
            selected_rounds = int(best_iteration) + 1

        row = {
            "snapshot": snapshot,
            "candidate": candidate_name,
            "validation_quarters": ",".join(
                str(int(q)) for q in validation_quarters
            ),
            "model_train_rows": len(X_model_train),
            "validation_rows": len(X_validation),
            "selected_boosting_rounds": selected_rounds,
            "validation_actual_latest_total": validation_latest_total,
            "validation_predicted_latest_total": (
                predicted_validation_latest_total
            ),
            "validation_latest_difference": (
                predicted_validation_latest_total
                - validation_latest_total
            ),
            "validation_latest_bias_ratio": validation_bias_ratio,
            "validation_abs_bias_deviation": (
                abs(validation_bias_ratio - 1.0)
                if pd.notna(validation_bias_ratio)
                else np.inf
            ),
            "validation_ay_wape": validation_ay_wape,
            "validation_claim_future_mae": mean_absolute_error(
                y_validation, predicted_validation_future
            ),
            "validation_claim_future_rmse": np.sqrt(
                mean_squared_error(
                    y_validation, predicted_validation_future
                )
            ),
            "validation_claim_latest_wape": wape(
                validation_actual_latest,
                validation_latest_deployed,
            ),
            "severe_threshold_training_p90": severe_threshold,
            **severe_result,
            **candidate_params,
        }

        candidate_rows.append(row)
        candidate_objects[candidate_name] = {
            "model": model,
            "preprocessor": preprocessor_selection,
            "row": row,
        }

        print(
            candidate_name,
            "| rounds:", selected_rounds,
            "| bias:", f"{validation_bias_ratio:.3f}",
            "| AY WAPE:", f"{validation_ay_wape:.3f}",
            "| severe WAPE:", f"{severe_result['severe_future_wape']:.3f}",
        )

    candidate_df = pd.DataFrame(candidate_rows)

    # Primary selection: lowest AY WAPE.
    # Tie band: within 0.002 AY WAPE, then choose smallest absolute bias.
    minimum_wape = candidate_df["validation_ay_wape"].min()
    tied = candidate_df[
        candidate_df["validation_ay_wape"] <= minimum_wape + 0.002
    ].copy()

    selected_row = (
        tied.sort_values(
            ["validation_abs_bias_deviation", "validation_ay_wape"]
        )
        .iloc[0]
    )
    selected_candidate = selected_row["candidate"]
    selected_rounds = int(selected_row["selected_boosting_rounds"])

    print(
        "Selected:", selected_candidate,
        "| rounds:", selected_rounds,
        "| validation bias:",
        f"{selected_row['validation_latest_bias_ratio']:.3f}",
        "| validation AY WAPE:",
        f"{selected_row['validation_ay_wape']:.3f}",
    )

    validation_results.extend(candidate_rows)

    # ------------------------------------------------------------------
    # Refit selected candidate on all training observations.
    # The test set remains untouched.
    # ------------------------------------------------------------------
    selected_candidate_params = next(
        {
            key: value for key, value in candidate.items()
            if key != "candidate"
        }
        for candidate in CANDIDATES
        if candidate["candidate"] == selected_candidate
    )

    preprocessor_final = build_preprocessor()
    X_train_full_processed = preprocessor_final.fit_transform(X_train_full)
    X_test_processed = preprocessor_final.transform(X_test)

    final_params = {
        **selected_candidate_params,
        "objective": "reg:squarederror",
        "booster": "gbtree",
        "tree_method": "hist",
        "n_estimators": selected_rounds,
        "eval_metric": "rmse",
        "random_state": 42,
        "n_jobs": -1,
        "verbosity": 0,
    }

    final_model = XGBRegressor(**final_params)
    final_model.fit(
        X_train_full_processed,
        targets_train_full[TARGET_FUTURE].astype(float).to_numpy(),
        verbose=False,
    )

    predicted_test_future_raw = final_model.predict(X_test_processed)
    test_snapshot_amount = (
        targets_test[SNAPSHOT_EXCESS].astype(float).to_numpy()
    )
    test_actual_future = (
        targets_test[TARGET_FUTURE].astype(float).to_numpy()
    )
    test_actual_latest = (
        targets_test[TARGET_LATEST].astype(float).to_numpy()
    )

    (
        test_latest_raw,
        test_latest_deployed,
        test_future_deployed,
    ) = reconstruct_latest(
        test_snapshot_amount,
        predicted_test_future_raw,
    )

    test_ay_summary, test_ay_wape = accident_year_summary(
        meta_test,
        test_snapshot_amount,
        test_actual_future,
        test_actual_latest,
        predicted_test_future_raw,
        test_latest_deployed,
    )

    test_actual_latest_total = float(test_actual_latest.sum())
    test_predicted_latest_total = float(test_latest_deployed.sum())
    test_bias_ratio = safe_ratio(
        test_predicted_latest_total,
        test_actual_latest_total,
    )

    test_severe = severe_metrics(
        test_actual_future,
        predicted_test_future_raw,
        severe_threshold,
    )

    test_result = {
        "model": "XGBoost",
        "snapshot": snapshot,
        "selected_candidate": selected_candidate,
        "selected_boosting_rounds": selected_rounds,
        "test_claims": len(X_test),
        "actual_snapshot_bixs_total": float(test_snapshot_amount.sum()),
        "actual_future_development_total": float(test_actual_future.sum()),
        "predicted_future_development_total_raw": float(
            predicted_test_future_raw.sum()
        ),
        "predicted_future_development_total_deployed": float(
            test_future_deployed.sum()
        ),
        "actual_latest_bixs_total": test_actual_latest_total,
        "predicted_latest_bixs_total": test_predicted_latest_total,
        "latest_difference_predicted_minus_actual": (
            test_predicted_latest_total - test_actual_latest_total
        ),
        "latest_bias_ratio": test_bias_ratio,
        "claim_level_future_mae": mean_absolute_error(
            test_actual_future, predicted_test_future_raw
        ),
        "claim_level_future_rmse": np.sqrt(
            mean_squared_error(
                test_actual_future, predicted_test_future_raw
            )
        ),
        "claim_level_latest_wape": wape(
            test_actual_latest, test_latest_deployed
        ),
        "accident_year_latest_wape": test_ay_wape,
        "severe_threshold_training_p90": severe_threshold,
        **test_severe,
    }
    final_test_results.append(test_result)

    prediction_df = meta_test.reset_index(drop=True).copy()
    prediction_df["snapshot"] = snapshot
    prediction_df["actual_snapshot_bixs"] = test_snapshot_amount
    prediction_df["actual_future_development"] = test_actual_future
    prediction_df["actual_latest_bixs"] = test_actual_latest
    prediction_df["predicted_future_development_raw"] = (
        predicted_test_future_raw
    )
    prediction_df["predicted_latest_bixs_raw"] = test_latest_raw
    prediction_df["predicted_latest_bixs"] = test_latest_deployed
    prediction_df["predicted_future_development_deployed"] = (
        test_future_deployed
    )
    prediction_df["latest_error"] = (
        prediction_df["predicted_latest_bixs"]
        - prediction_df["actual_latest_bixs"]
    )
    prediction_df["latest_absolute_error"] = (
        prediction_df["latest_error"].abs()
    )
    prediction_df["is_severe_future_outcome"] = (
        prediction_df["actual_future_development"] >= severe_threshold
    )

    test_ay_summary["model"] = "XGBoost"
    test_ay_summary["snapshot"] = snapshot
    final_ay_results.append(test_ay_summary)

    snapshot_output = OUTPUT_FOLDER / snapshot
    snapshot_output.mkdir(parents=True, exist_ok=True)

    candidate_df.to_csv(
        snapshot_output / "xgb_validation_candidates.csv",
        index=False,
    )
    prediction_df.to_parquet(
        snapshot_output / "xgb_test_predictions.parquet",
        index=False,
    )
    test_ay_summary.to_csv(
        snapshot_output / "xgb_test_predictions_by_accident_year.csv",
        index=False,
    )

    joblib.dump(
        {
            "preprocessor": preprocessor_final,
            "model": final_model,
            "selected_candidate": selected_candidate,
            "selected_boosting_rounds": selected_rounds,
            "final_parameters": final_params,
            "feature_manifest": manifest,
            "severe_threshold_training_p90": severe_threshold,
        },
        snapshot_output / f"xgb_future_development_{snapshot}.joblib",
    )

    # Gain-based feature importance.
    feature_names = preprocessor_final.get_feature_names_out()
    booster_scores = final_model.get_booster().get_score(
        importance_type="gain"
    )
    feature_importance = pd.DataFrame(
        {
            "feature": feature_names,
            "gain": [
                booster_scores.get(f"f{index}", 0.0)
                for index in range(len(feature_names))
            ],
        }
    ).sort_values("gain", ascending=False)

    total_gain = feature_importance["gain"].sum()
    feature_importance["gain_share"] = np.where(
        total_gain > 0,
        feature_importance["gain"] / total_gain,
        0.0,
    )
    feature_importance.to_csv(
        snapshot_output / "xgb_feature_importance_gain.csv",
        index=False,
    )

    selected_models[snapshot] = {
        "selected_candidate": selected_candidate,
        "selected_boosting_rounds": selected_rounds,
        "final_parameters": final_params,
        "validation_quarters": validation_quarters.tolist(),
        "severe_threshold_training_p90": severe_threshold,
    }

    print(
        "TEST",
        snapshot,
        "| actual latest:", f"{test_actual_latest_total:,.2f}",
        "| predicted latest:", f"{test_predicted_latest_total:,.2f}",
        "| bias:", f"{test_bias_ratio:.3f}",
        "| AY WAPE:", f"{test_ay_wape:.3f}",
        "| severe WAPE:", f"{test_severe['severe_future_wape']:.3f}",
    )

## Save XGBoost summaries

This cell collates the final XGBoost outputs so they can be checked, compared with Random Forest and Chain Ladder, and used in the dissertation results.

In [ ]:
validation_results_df = pd.DataFrame(validation_results)
xgb_test_summary_df = pd.DataFrame(final_test_results)
xgb_ay_summary_df = pd.concat(final_ay_results, ignore_index=True)

validation_results_df.to_csv(
    OUTPUT_FOLDER / "xgb_validation_candidate_summary.csv",
    index=False,
)
xgb_test_summary_df.to_csv(
    OUTPUT_FOLDER / "xgb_clean_test_summary_by_snapshot.csv",
    index=False,
)
xgb_ay_summary_df.to_csv(
    OUTPUT_FOLDER / "xgb_clean_test_summary_by_snapshot_and_accident_year.csv",
    index=False,
)

with open(OUTPUT_FOLDER / "xgb_selected_models.json", "w") as f:
    json.dump(selected_models, f, indent=2)

display(xgb_test_summary_df)

## Recalculate Random Forest severe-outcome metrics

The corrected Random Forest test predictions are evaluated using the same
training-derived severe threshold as XGBoost. This makes the severe-outcome
comparison like-for-like.

In this section, the Random Forest results are added to the XGBoost notebook so both models can be compared using the same severe-claim measure. For each snapshot, the RF test predictions are loaded, the same severe-claim threshold as XGBoost is applied, and MAE, RMSE, and WAPE are calculated for those severe claims. These results are then merged into the RF summary table.

In [ ]:
rf_summary_path = RF_OUTPUT_FOLDER / "rf_clean_test_summary_by_snapshot.csv"
if not rf_summary_path.exists():
    raise FileNotFoundError(
        f"Corrected RF summary not found: {rf_summary_path}"
    )

rf_summary = pd.read_csv(rf_summary_path)
rf_severe_rows = []

for snapshot in SNAPSHOTS:
    rf_prediction_path = (
        RF_OUTPUT_FOLDER
        / snapshot
        / "rf_test_predictions.parquet"
    )
    if not rf_prediction_path.exists():
        raise FileNotFoundError(
            f"Corrected RF prediction file not found: {rf_prediction_path}"
        )

    rf_predictions = pd.read_parquet(rf_prediction_path)
    threshold = severe_thresholds[snapshot]

    severe_result = severe_metrics(
        rf_predictions["actual_future_development"],
        rf_predictions["predicted_future_development_raw"],
        threshold,
    )

    rf_severe_rows.append({
        "model": "Random Forest",
        "snapshot": snapshot,
        "severe_threshold_training_p90": threshold,
        **severe_result,
    })

rf_severe_df = pd.DataFrame(rf_severe_rows)
rf_summary = rf_summary.merge(
    rf_severe_df,
    on=["model", "snapshot"],
    how="left",
)

display(rf_summary)

## Final Random Forest versus XGBoost comparison

This section builds a table to directly compare Random Forest and XGBoost. It uses the same performance measures for both models, puts them together in one table, sorts the results by snapshot and model, and saves the comparison as a CSV file.

In [ ]:
comparison_columns = [
    "model",
    "snapshot",
    "test_claims",
    "actual_snapshot_bixs_total",
    "actual_future_development_total",
    "predicted_future_development_total_raw",
    "actual_latest_bixs_total",
    "predicted_latest_bixs_total",
    "latest_difference_predicted_minus_actual",
    "latest_bias_ratio",
    "claim_level_future_mae",
    "claim_level_future_rmse",
    "claim_level_latest_wape",
    "accident_year_latest_wape",
    "severe_threshold_training_p90",
    "severe_claim_count",
    "severe_future_mae",
    "severe_future_rmse",
    "severe_future_wape",
]

rf_for_comparison = rf_summary[comparison_columns].copy()
xgb_for_comparison = xgb_test_summary_df[comparison_columns].copy()

model_comparison = pd.concat(
    [rf_for_comparison, xgb_for_comparison],
    ignore_index=True,
).sort_values(["snapshot", "model"])

model_comparison.to_csv(
    OUTPUT_FOLDER / "rf_vs_xgb_clean_test_comparison.csv",
    index=False,
)

display(model_comparison)

This section organizes the RF-versus-XGBoost comparison into a wide table, placing the Random Forest and XGBoost results side by side for each measure. Next, it calculates the difference between XGBoost and Random Forest for each metric, saves the table, and shows the final comparison.

In [ ]:
comparison_wide = model_comparison.pivot(
    index="snapshot",
    columns="model",
    values=[
        "predicted_latest_bixs_total",
        "latest_difference_predicted_minus_actual",
        "latest_bias_ratio",
        "claim_level_future_mae",
        "claim_level_future_rmse",
        "claim_level_latest_wape",
        "accident_year_latest_wape",
        "severe_future_mae",
        "severe_future_rmse",
        "severe_future_wape",
    ],
)

comparison_wide.columns = [
    f"{metric}_{model.replace(' ', '_')}"
    for metric, model in comparison_wide.columns
]
comparison_wide = comparison_wide.reset_index()

for metric in [
    "predicted_latest_bixs_total",
    "latest_difference_predicted_minus_actual",
    "latest_bias_ratio",
    "claim_level_future_mae",
    "claim_level_future_rmse",
    "claim_level_latest_wape",
    "accident_year_latest_wape",
    "severe_future_mae",
    "severe_future_rmse",
    "severe_future_wape",
]:
    rf_col = f"{metric}_Random_Forest"
    xgb_col = f"{metric}_XGBoost"
    if rf_col in comparison_wide.columns and xgb_col in comparison_wide.columns:
        comparison_wide[f"{metric}_XGB_minus_RF"] = (
            comparison_wide[xgb_col] - comparison_wide[rf_col]
        )

comparison_wide.to_csv(
    OUTPUT_FOLDER / "rf_vs_xgb_clean_test_comparison_wide.csv",
    index=False,
)

display(comparison_wide)
print("Saved all XGBoost and comparison outputs to:", OUTPUT_FOLDER)

This section builds a clear calibration table for the XGBoost models. It takes the final model settings chosen for each snapshot from the saved JSON file, pulls out key hyperparameters like boosting rounds, learning rate, tree depth, regularisation, and subsampling, and combines them in one table.

In [ ]:
from pathlib import Path
import json
import pandas as pd

OUTPUT_FOLDER = Path(
    "/path/to/"
    "BI_large_claims_project/processed/chapter5_outputs/"
    "section_5_8A_clean_xgboost_future_development"
)

json_path = OUTPUT_FOLDER / "xgb_selected_models.json"

if not json_path.exists():
    raise FileNotFoundError(
        f"Could not find {json_path}. Run the full XGBoost notebook first."
    )

with open(json_path, "r") as f:
    selected_models = json.load(f)

rows = []

for snapshot, details in selected_models.items():
    params = details["final_parameters"]

    rows.append({
        "Snapshot": snapshot,
        "Selected candidate": details["selected_candidate"],
        "Boosting rounds": details["selected_boosting_rounds"],
        "Learning rate": params["learning_rate"],
        "Maximum depth": params["max_depth"],
        "Minimum child weight": params["min_child_weight"],
        "Row subsample": params["subsample"],
        "Feature subsample": params["colsample_bytree"],
        "L1 regularisation": params["reg_alpha"],
        "L2 regularisation": params["reg_lambda"],
        "Minimum loss reduction": params["gamma"],
        "Objective": params["objective"],
        "Tree method": params["tree_method"],
        "Evaluation metric": params["eval_metric"],
        "Random seed": params["random_state"],
    })

xgb_calibration_table = pd.DataFrame(rows)

display(xgb_calibration_table)

xgb_calibration_table.to_csv(
    OUTPUT_FOLDER / "xgb_dissertation_calibration_table.csv",
    index=False,
)

print(
    xgb_calibration_table.to_latex(
        index=False,
        escape=True,
        float_format="%.3f"
    )
)